Finding the best site the bet on is a time-consuming and challenging task. With odds constantly changing, finding what place is giving you the best odds manually is nearly impossible. Initially, our objective was to create a web-scraping algorithm to scrape odds from different betting sites in order to obtain our information. We quickly realized that this process would be too difficult since 1. betting sites encrypt their data when you inspect their page to prevent web scraping and 2. if we were to manage to webscrape a betting site, many betting sites update their encryptions periodically which could prove to be another problem down the road. 

Instead, we found a sports betting API (the-odds-api) that compiles the moneyline and spread odds from multiple sports betting sites. Due to the limited number of games per day, we want to create a pipeline that cleans the data we retrieve as a json into a more readable nested dictionary/pandas dataframe. 

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json

# Source: https://the-odds-api.com/
#url = f'https://api.the-odds-api.com/v4/sports/basketball_nba/odds/?apiKey=eaa265db381b34b1dbe2ce35e2628048&regions=us&markets=h2h,spreads,totals&oddsFormat=american'
#betting_text = requests.get(url).text
betting_data = json.loads(betting_text)

In [4]:
len(betting_data)

5

In [21]:
sites_list = list()
for game in betting_data:
    bookmakers = game['bookmakers']
    for site in bookmakers:
        sites_list.append(site['title'])

In [28]:
sites_list = list(set(sites_list))

In [27]:
betting_data

[{'id': '3259d402526bf11efb5d6bac201cc631',
  'sport_key': 'basketball_nba',
  'sport_title': 'NBA',
  'commence_time': '2022-03-29T23:10:00Z',
  'home_team': 'Washington Wizards',
  'away_team': 'Chicago Bulls',
  'bookmakers': [{'key': 'foxbet',
    'title': 'FOX Bet',
    'last_update': '2022-03-29T21:52:49Z',
    'markets': [{'key': 'h2h',
      'outcomes': [{'name': 'Chicago Bulls', 'price': -200},
       {'name': 'Washington Wizards', 'price': 145}]},
     {'key': 'spreads',
      'outcomes': [{'name': 'Chicago Bulls', 'price': -118, 'point': -4.5},
       {'name': 'Washington Wizards', 'price': 100, 'point': 4.5}]},
     {'key': 'totals',
      'outcomes': [{'name': 'Over', 'price': -110, 'point': 223.0},
       {'name': 'Under', 'price': -110, 'point': 223.0}]}]},
   {'key': 'draftkings',
    'title': 'DraftKings',
    'last_update': '2022-03-29T21:52:36Z',
    'markets': [{'key': 'h2h',
      'outcomes': [{'name': 'Chicago Bulls', 'price': -180},
       {'name': 'Washington Wi

In [32]:
data_by_game = dict()

for game in betting_data:
    # Get the home team and away team by game and format it for easier viewing
    home = game['home_team']
    away = game['away_team']
    matchup = f'{away} @ {home}'
    
    # Keep the matchup name as the key, will store odds by site as value
    data_by_game[matchup] = dict()
    
    bookmakers = game['bookmakers']
    for site in bookmakers:
        site_title = site['title']
        data_by_game[matchup][site_title] = dict()
        markets = site['markets']
        
        for market in markets:
            bet_format = market['key']
            data_by_game[matchup][site_title][bet_format] = market['outcomes']

In [33]:
data_by_game

{'Chicago Bulls @ Washington Wizards': {'FOX Bet': {'h2h': [{'name': 'Chicago Bulls',
     'price': -200},
    {'name': 'Washington Wizards', 'price': 145}],
   'spreads': [{'name': 'Chicago Bulls', 'price': -118, 'point': -4.5},
    {'name': 'Washington Wizards', 'price': 100, 'point': 4.5}],
   'totals': [{'name': 'Over', 'price': -110, 'point': 223.0},
    {'name': 'Under', 'price': -110, 'point': 223.0}]},
  'DraftKings': {'h2h': [{'name': 'Chicago Bulls', 'price': -180},
    {'name': 'Washington Wizards', 'price': 155}],
   'spreads': [{'name': 'Chicago Bulls', 'price': -110, 'point': -4.5},
    {'name': 'Washington Wizards', 'price': -110, 'point': 4.5}],
   'totals': [{'name': 'Over', 'price': -110, 'point': 223.5},
    {'name': 'Under', 'price': -110, 'point': 223.5}]},
  'FanDuel': {'h2h': [{'name': 'Chicago Bulls', 'price': -198},
    {'name': 'Washington Wizards', 'price': 166}],
   'spreads': [{'name': 'Chicago Bulls', 'price': -110, 'point': -4.5},
    {'name': 'Washington